# Stage 6: LoRA (QLoRA 4-bit)

Bases: Qwen2.5-Coder-7B-Instruct and the best open 14B model in the stage-3 top 3. Variants L1 (BIRD train),
L2 at 50/100/200/400 synthetic pairs, L3 (L1 then L2-400). Checkpoints every 0.5 epoch, best picked on the
validation set. Training needs `uv sync --extra train` and the GPU (stop any vLLM server first).

In [ ]:
from pathlib import Path
import json, os, subprocess
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from t2sbench.bench import load_results
from t2sbench.report import aggregate
pd.set_option("display.max_colwidth", 120)
RUN = False  # set True to execute the stage from here (it can take hours and cost money)

def sh(cmd):
    print("$", cmd)
    if RUN:
        subprocess.run(cmd, shell=True, check=True)

def summary(stage):
    df = load_results(Path("results"), stage)
    if df.empty:
        print(f"no results for {stage} yet")
        return df
    agg = aggregate(df)
    return agg[["model", "adapter", "strategy", "dataset", "n", "ex", "ci_lo", "ci_hi", "syntax_ok", "no_answer",
                "time_median", "time_p90", "in_tokens_mean", "out_tokens_mean", "cost_per_100"]].sort_values(
                ["dataset", "ex"], ascending=[True, False])

In [ ]:
sh("uv run t2sbench stage 6-train")

In [ ]:
metas = [json.loads(p.read_text()) for p in sorted(Path("adapters").glob("*/train_meta.json"))]
pd.DataFrame(metas)[["name", "train_examples", "dropped_too_long", "best_checkpoint", "best_val_ex", "train_seconds", "adapter_size_bytes"]] if metas else "no adapters yet"

In [ ]:
sh("uv run t2sbench stage 6-eval")

In [ ]:
summary("stage6")